# dARK Core Minter API Test

This notebook demonstrates the lifecycle of ARK identifiers using the new Minter API endpoints. It is self-contained and includes authority registration via the Admin API.

In [ ]:
import requests
import json
import uuid
import time

ADMIN_URL = "http://localhost:8000/api/v1/admin"
MINTER_URL = "http://localhost:8001/api/v1"

# Unique IDs for this run
AUTHORITY_ID = f"test-authority-{int(time.time())}"
NAAN = "12345"

## 0. Setup Authority
Register the authority and authorize the NAAN using the Admin API.

In [ ]:
print(f"Registering authority: {AUTHORITY_ID}...")
reg_payload = {
    "uuid": AUTHORITY_ID,
    "naans": [NAAN],
    "fund_amount_eth": 0.05
}

try:
    response = requests.post(f"{ADMIN_URL}/authority", json=reg_payload)
    print(f"Status: {response.status_code}")
    print(json.dumps(response.json(), indent=2))
except Exception as e:
    print(f"Error: {e}. Is the Admin API running on port 8000?")

## 1. Reserve Single ARK

In [ ]:
payload = {
    "authority_id": AUTHORITY_ID,
    "naan": NAAN,
    "alternate_identifiers": [
        {"schema": "doi", "value": f"10.1000/test-{uuid.uuid4().hex[:8]}"}
    ]
}

response = requests.post(f"{MINTER_URL}/arks", json=payload)
print(json.dumps(response.json(), indent=2))
ark_id = response.json().get("ark")

## 2. Publish ARK (Update Metadata)

In [ ]:
update_payload = {
    "authority_id": AUTHORITY_ID,
    "target": "https://example.com/notebook-test",
    "metadata": {
        "title": "Notebook Test Record",
        "creator": "Jupyter"
    },
    "alternate_identifiers": [
        {"schema": "doi", "value": "10.1000/test-notebook"},
        {"schema": "internal", "value": "nb-001"}
    ]
}

response = requests.put(f"{MINTER_URL}/arks/{ark_id}", json=update_payload)
print(json.dumps(response.json(), indent=2))

## 3. Resolve ARK

In [ ]:
response = requests.get(f"{MINTER_URL}/arks/{ark_id}")
print(json.dumps(response.json(), indent=2))

## 4. Batch Reserve

In [ ]:
batch_payload = {
    "authority_id": AUTHORITY_ID,
    "naan": NAAN,
    "items": [
        {"target": "https://example.com/b1", "alternate_identifiers": [{"schema": "idx", "value": "1"}], "client_item_id": "req-1"},
        {"target": "https://example.com/b2", "client_item_id": "req-2"}
    ]
}

response = requests.post(f"{MINTER_URL}/arks/batch", json=batch_payload)
print(json.dumps(response.json(), indent=2))

## 5. Tombstone

In [ ]:
response = requests.delete(f"{MINTER_URL}/arks/{ark_id}")
print(f"Status: {response.status_code}")